In [ ]:
import getpass
from pathlib import Path
from Physical_Quantity.Physical_Data_Record import SuperCorrRecord
from Physical_Quantity.Super_Correlation_Process import SuperCorrProcessor

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#-----------------########### UltraDMRG计算的三个键方向的超导关联函数<Δ_ij^†·Δ_kl>数据提取与作图 ################----------------------
### 基本参数
N = 200
a = 1.0

### ================= 文件夹路径变量（需要修改的部分）=====================
δ = "d1_10"               # 空穴率
version = "50_54"         # reference bond
bond_dim = "1500"         # bond dimension
t_J = "t_J_3"             # t/J

base_dir = Path(f'Data_N{N}/{t_J}/{δ}/{bond_dim}_measure_super_corr/super_corr_{version}')

### 将三个键方向A、B、C的数据提取并封装成SuperCorrRecord对象列表
records = {}
for bond in 'ABC':
    for channel in 'abcd':
        key = f'scs{bond}{channel}'
        fname = f'{key}_fix_refbond_{version}.json'
        records[key] = SuperCorrRecord.extract_data(
            (base_dir / fname).read_text(encoding='utf-8'), N, a
        )

### 方向显示名映射字典
bond_labels = {
    'A': "A key (blue bond)",
    'B': "B key (green bond)",
    'C': "C key (orange bond)",
}

### 检测超导关联对称性并组合超导关联函数
super_corr_dict_UltraDMRG = {}
for bond, label in bond_labels.items():
    ## 取出当前 bond 对应的四个 SuperCorrRecord 对象
    recs = [records[f'scs{bond}{channel}'] for channel in 'abcd']

    ## 对称性检查
    err_ad, err_bc = SuperCorrProcessor.symmetry_check(*recs)
    print(f"{bond}键方向的通道a与通道d的相对误差为：{err_ad}；通道b与通道c的相对误差为：{err_bc}")

    ## 组合超导关联函数，存入最终字典
    super_corr_dict_UltraDMRG[label] = SuperCorrProcessor.Combined_super_correlation(*recs)

### 可视化
SuperCorrProcessor.Super_Corr_Visualization_three(super_corr_dict_UltraDMRG)

In [ ]:
#---------------------------------------------------------------------------------------------------------------------------------
#---------------------########### Itensor计算超导关联函数<Δ_ij^†·Δ_kl>数据提取与作图 ################--------------------------
### 基本参数
N = 200  # 格点数
a = 1.0  # honeycomb晶格常数

### 三个方向与可视化的label的映射
directions = {
    'A': "A key (blue bond)",
    'B': "B key (green bond)",
    'C': "C key (orange bond)"
}

### 输入数据并封装成SuperCorrRecord对象
super_corr_dict_Itensor = {}
for bond, label in directions.items():
    data = getpass.getpass(f"请输入文件scs_fix_{bond}的数据: ")
    super_corr_dict_Itensor[label] = SuperCorrRecord.extract_data(data, N, a)

### 数据可视化
SuperCorrProcessor.Super_Corr_Visualization_three(super_corr_dict_Itensor)

In [ ]:
#--------------------------------------------------------------------------------------------------------------------------
#---------------------########### UltraDMRG相对于Itensor的超导关联函数的绝对误差 ################--------------------------

### UltraDMRG相对于Itensor的超导关联函数的绝对误差可视化
directions = {
    'UltraDMRG': super_corr_dict_UltraDMRG,
    'Itensor': super_corr_dict_Itensor
}
SuperCorrProcessor.Super_Corr_Visualization_difference(directions)

In [ ]:
#-------------------------------------------------------------------------------------------------------------------------
#---------------------########### UltraDMRG在不同bond下的超导关联函数的绝对误差 ################--------------------------

### 基本参数
N = 200
a = 1.0

### ================= 文件夹路径变量（需要修改的部分）=====================
δ = "d1_10"                              # 空穴率
bond_dim1 = "1500"                        # bond_dim1
bond_dim2 = "5000"                        # bond_dim2
t_J = "t_J_3"                             # t/J
versions = ["49_50", "50_51", "50_54"]    # reference bond


#-------------------------------------------------------------------------------------------------------------------------
#--------------------------------------- 数据读取与组合 -----------------------------------------------------------------

def load_super_corr(version):

    # 文件夹路径
    base_dirs = {
        bond_dim1: Path(
            f'Data_N{N}/{t_J}/{δ}/{bond_dim1}_measure_super_corr/super_corr_{version}'
        ),
        bond_dim2: Path(
            f'Data_N{N}/{t_J}/{δ}/{bond_dim2}_measure_super_corr/super_corr_{version}'
        ),
    }

    # 读取并封装
    records = {
        bond_dim: {
            key: SuperCorrRecord.extract_data(
                (base_dir / f'{key}_fix_refbond_{version}.json').read_text(encoding='utf-8'), N, a
            )
            for key in [
                'scsAa', 'scsAb', 'scsAc', 'scsAd',
                'scsBa', 'scsBb', 'scsBc', 'scsBd',
                'scsCa', 'scsCb', 'scsCc', 'scsCd'
            ]
        }
        for bond_dim, base_dir in base_dirs.items()
    }

    # 组合超导关联函数
    combined = {
        bond_dim: {
            key: SuperCorrProcessor.Combined_super_correlation(
                *(records[bond_dim][f"scs{key}{s}"] for s in "abcd")
            )
            for key in "ABC"
        }
        for bond_dim in base_dirs
    }

    # 构造可视化需要的字典
    return {
        f"bond dim: {bond_dim1}": {
            "A key (blue bond)": combined[bond_dim1]["A"],
            "B key (green bond)": combined[bond_dim1]["B"],
            "C key (orange bond)": combined[bond_dim1]["C"],
        },
        f"bond dim: {bond_dim2}": {
            "A key (blue bond)": combined[bond_dim2]["A"],
            "B key (green bond)": combined[bond_dim2]["B"],
            "C key (orange bond)": combined[bond_dim2]["C"],
        },
    }


#-------------------------------------------------------------------------------------------------------------------------
#--------------------------------------- 依次计算三个 reference bond 的绝对误差 -----------------------------------------
### 将多个reference bond(存储在versions中)在不同bond-dim下进行绝对误差可视化
for version in versions:
    ## 调用上面定义的函数load_super_corr进行数据读取与组合
    directions_bond = load_super_corr(version)
    
    ## 可视化
    SuperCorrProcessor.Super_Corr_Visualization_difference(
        directions_bond
    )